# Realistic clouds from observations (thermal)

Builds cloud layers from lidar and radar observations of HALO-(AC)³ research flight RF13 (2022-04-04) and compares the simulated 10.74 µm nadir brightness temperature with VELOX channel 3.

```{note}
This notebook needs HALO-(AC)³ campaign data from `/projekt_agmwend` (VELOX, lidar, radar) and the dropsondes in `radiosonde/`, none of which are public. It is not executed for the book and has no stored outputs; run it with the campaign data, or replace the file paths with your own observations.
```

In [ ]:
import pyradtran
from pyradtran import load_config
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from pathlib import Path
import logging

logging.getLogger('pyradtran').setLevel(logging.CRITICAL)

cfg = load_config()
cfg.simulation_defaults.rte_solver = "disort"
cfg.simulation_defaults.mol_abs_param = "reptran medium"
cfg.simulation_defaults.source = "thermal"
cfg.simulation_defaults.wavelength_nm = [10350, 11130]    # VELOX channel 3 (10.74 µm)
cfg.simulation_defaults.integrate_wavelength = True
cfg.simulation_defaults.albedo_value = 0.01               # emissivity 0.99 (snow-covered sea ice)
cfg.simulation_defaults.surface_temperature_k = 273.15    # assumption; adjust to the scene
cfg.simulation_defaults.output_columns = ["zout", "lambda", "eup", "edn", "uu"]
cfg.simulation_defaults.h2o_source = "radiosonde"
cfg.simulation_defaults.ozone_du = 300.0
cfg.paths.radiosonde_base = "radiosonde"                  # dropsondes, not shipped
cfg.execution.max_workers = 8
cfg.execution.timeout_seconds = 120
cfg.output.filename_prefix = "radiosonde_thermal"
config_path = Path("config/radiosonde_thermal.yaml")
cfg.to_yaml(config_path)

## Load observation data

VELOX channel 3 (10.74 µm) brightness temperatures from RF13. `BT_2D` and `BT_Center` are in °C.

In [ ]:
ds = xr.open_dataset(
    "/projekt_agmwend/data/HALO-AC3/02_Flights/HALO-AC3_20220404_HALO_RF13/VELOX/VELOX_327kveL/Processed/to_publish/final/HALO-AC3_HALO_VELOX_BT3_10740nm_20220404_RF13_v3.0.nc"
).sel(time=slice('2022-04-04T13:41:00', '2022-04-04T14:00:00')).load()

ds.BT_Center.plot(figsize=(20, 3))

In [ ]:
# One VELOX frame and its brightness-temperature histogram
time_idx = 240

fig, ax = plt.subplots(1, 2, figsize=(7, 3))
bins = np.linspace(-25, 0, 100)
ds.BT_2D.isel(time=time_idx).plot(cmap='viridis', vmin=-25, vmax=0, ax=ax[1],
                                  cbar_kwargs={'label': 'Brightness temperature (°C)'})
ds.BT_2D.isel(time=time_idx).plot.hist(ax=ax[0], bins=bins, color='gray', alpha=0.7, histtype='step', lw=2)
ax[0].set_xlabel('Brightness temperature (°C)')
ax[0].set_ylabel('Count')
plt.tight_layout()
plt.show()

## Prepare the flight track

Only the track goes into the simulation: every dimension of the input dataset is a loop, so the image dimensions `x`, `y` must not be passed. The aircraft altitude is a data variable, so each time step is simulated at flight level ({doc}`/guide/concepts`).

In [ ]:
track = xr.Dataset({
    'latitude': ds['lat'],
    'longitude': ds['lon'],
    'altitude': ds['alt'] / 1000,   # m -> km, flight level per time step
})
track

## Clear-sky simulation

Nadir brightness temperature (`uu`) at flight level without clouds.

In [ ]:
ds_sim_clear = track.pyradtran.run(
    config_path=config_path,
    save_to_file=False,
    params={'output_quantity': 'brightness'},
    show_progress=False,
)
ds_sim_clear

## Cloud layers from lidar and radar

Cloud top and base come from the lidar backscatter ratio (> 100 counts as cloud); the radar reflectivity is shown for context. Each cloudy time step gets one homogeneous liquid layer with 0.1 g m⁻³ and the default effective radius of 10 µm ({doc}`/guide/clouds`). The water content is an assumption, not a retrieval.

In [ ]:
lidar = xr.open_dataset("/projekt_agmwend/home_rad/Joshua/HALO-AC3_unified_data/unified_bsrgl.nc").sel(time=ds.time).load()
lidar = lidar.where(lidar.backscatter_ratio > 100)
radar = xr.open_dataset("/projekt_agmwend/home_rad/Joshua/HALO-AC3_unified_data/unified_radar.nc").sel(time=ds.time).load()

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 3), gridspec_kw={'width_ratios': [1, .2]})
lidar.backscatter_ratio.plot(y='altitude', robust=True, add_colorbar=False, ax=ax)
radar.dBZg.plot(y='height', robust=True, cmap='gray_r', alpha=.7, add_colorbar=False, ax=ax)
ax.set_ylim(0, 2000)


def cth_and_layer_thickness(ds, ds_lidar=lidar):
    """Cloud top height, thickness (m) and cloud fraction from lidar data
    that are already masked to backscatter_ratio > 100."""
    ds_lidar_sel = ds_lidar.sel(time=ds.time, method='nearest')
    cloudy = ds_lidar_sel.backscatter_ratio.notnull()
    cth = ds_lidar_sel.altitude.where(cloudy).max(dim='altitude')
    cbh = ds_lidar_sel.altitude.where(cloudy).min(dim='altitude')
    # Ignore returns below 150 m (surface)
    cth = cth.where(cth > 150)
    cbh = cbh.where(cbh > 150)
    cloudmask = (cloudy & (ds_lidar_sel.altitude > 150)).mean(dim='altitude')
    return xr.Dataset({'cth': cth, 'thickness': cth - cbh, 'cloudmask': cloudmask})


cth_ds = cth_and_layer_thickness(ds).dropna('time')
cth = cth_ds.cth

# Smooth over 5 time steps, at least 100 m thick, cloudy steps only
cth_interp = cth.rolling(time=5, center=True, min_periods=1).mean().interpolate_na('time')
thickness_interp = cth_ds.thickness.rolling(time=5, center=True, min_periods=1).mean().interpolate_na('time')
thickness_interp = thickness_interp.where(thickness_interp > 100, 100)
cloudy_steps = cth_ds.cloudmask > 0
thickness_interp = thickness_interp.sel(time=cloudy_steps)
cth_interp = cth_interp.sel(time=cloudy_steps).fillna(500)   # 500 m where no cloud top is left

cth_interp.plot(ax=ax, label='Cloud top height', color='blue')
(cth_interp - thickness_interp).plot(ax=ax, label='Cloud base height', color='orange')
ax.set_ylabel('Altitude (m)')
ax.legend()
ax2.hist(cth, bins=25)
ax2.set_xlabel('Cloud top height (m)')

## Cloudy simulation

One batch run: cloud top, base and water content are dataset variables (`cloud_top_var`, `cloud_bottom_var`, `cloud_wc_var`), so every time step gets its own layer.

In [ ]:
cloudy = track.sel(time=cth_interp.time)
cloudy['cth'] = cth_interp / 1000                         # km
cloudy['cbh'] = (cth_interp - thickness_interp) / 1000    # km
cloudy['lwc'] = xr.full_like(cloudy['cth'], 0.1)          # g m-3

print(cloudy.pyradtran.inspect_cloud_file(selector={'time': cloudy.time.values[0]},
                                          cloud_wc_var='lwc', cloud_top_var='cth', cloud_bottom_var='cbh'))

ds_sim_cloudy = cloudy.pyradtran.run(
    config_path=config_path,
    save_to_file=False,
    params={'output_quantity': 'brightness'},
    cloud_wc_var='lwc',
    cloud_top_var='cth',
    cloud_bottom_var='cbh',
    show_progress=False,
)
ds_sim_cloudy

## Cloud threshold

VELOX pixels colder than the median brightness temperature of all frames count as cloudy.

In [ ]:
ds.BT_2D.plot.hist(bins=100)
median = float(ds.BT_2D.quantile(0.5, ['x', 'y', 'time']).values)
plt.axvline(median, color='red', linestyle='--', label='Median BT')
plt.xlabel('Brightness temperature (°C)')
plt.legend()
print(f"Median brightness temperature: {median:.2f} °C ({median + 273.15:.2f} K)")

ds_cloudy = ds.sel(time=cloudy.time).where(ds.BT_2D < median)

## Simulated against measured brightness temperature

Simulated nadir brightness temperature against the mean of the cloudy VELOX pixels in the central 100 × 100 pixels.

In [ ]:
window = 50
y_min, y_max = 250, 270

sim_bt = ds_sim_cloudy.uu.isel(altitude=0)                       # K
obs_bt = ds_cloudy.BT_2D.isel(
    x=slice(635 // 2 - window, 635 // 2 + window),
    y=slice(507 // 2 - window, 507 // 2 + window),
).mean(['x', 'y']) + 273.15                                         # °C -> K

fig, (ax_hists, ax, ax_scatter) = plt.subplots(1, 3, figsize=(18, 3), gridspec_kw={'width_ratios': [.3, 1, .3]})

sim_bt.plot(marker='o', color='blue', label='Simulated', ax=ax)
obs_bt.plot(label='Measured (cloudy pixels)', color='red', linestyle='--', marker='x', ax=ax)
ax.set_xlabel('Time (UTC)')
ax.set_ylabel('Brightness temperature 10.74 µm (K)')
ax.set_ylim(y_min, y_max)

X, Y = sim_bt.values, obs_bt.values
ok = np.isfinite(X) & np.isfinite(Y)
X, Y = X[ok], Y[ok]
bins = np.arange(y_min, y_max, 1)
h = ax_scatter.hist2d(X, Y, bins=bins, cmin=1, cmap='viridis')
plt.colorbar(h[3], ax=ax_scatter, label='Counts')
ax_scatter.plot([y_min, y_max], [y_min, y_max], color='black', linestyle='--', alpha=0.5, label='1:1')
ax_scatter.set_xlabel('Simulated BT (K)')
ax_scatter.set_ylabel('Measured BT (K)')

rmse = np.sqrt(np.mean((X - Y) ** 2))
mae = np.mean(np.abs(X - Y))
bias = np.mean(X - Y)
r2 = np.corrcoef(X, Y)[0, 1] ** 2
ax_scatter.text(0.6, 0.3, f'RMSE: {rmse:.2f} K\nMAE: {mae:.2f} K\nBias: {bias:.2f} K\nR²: {r2:.2f}',
                transform=ax_scatter.transAxes, fontsize=10, verticalalignment='top')

ax_hists.hist(X, bins=bins, color='blue', alpha=0.5, label='Simulated')
ax_hists.hist(Y, bins=bins, color='red', alpha=0.5, label='Measured')
ax_hists.set_xlabel('Brightness temperature (K)')

for a in (ax, ax_scatter, ax_hists):
    a.spines[['top', 'right']].set_visible(False)
    a.grid(True, linestyle='--', alpha=0.5)
    a.legend()
    a.set_title('')